# L56 · 误差向后 — 反向传播手推

**目标**：实现 `Value.backward()`，从输出节点出发，拓扑排序后逐节点调用 `_backward()`，把梯度沿计算图向后传播。

🔗 **Aurora 连接**：这正是 PyTorch 的 `.backward()` 在做的事情；Month 2 训练循环（权重更新）依赖此机制。

前两节课我们给每个运算附上了局部的 `_backward()` 函数，知道如何把"上游梯度"分发给当前节点的两个输入。
这节课解决剩下的问题：**用什么顺序**依次调用所有节点的 `_backward()`？
答案是拓扑排序——保证每个节点在其所有"下游消费者"都处理完之前不动，从而让梯度能正确地从输出一路积累回输入。

In [ ]:
import numpy as np

# ── Value 基础实现（含各运算的 _backward，backward 留待下方实现）──
class Value:
    def __init__(self, data, _children=(), _op=''):
        self.data = float(data)
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')
        def _bwd():
            self.grad  += out.grad
            other.grad += out.grad
        out._backward = _bwd
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')
        def _bwd():
            self.grad  += other.data * out.grad
            other.grad += self.data  * out.grad
        out._backward = _bwd
        return out

    def __neg__(self):        return self * -1
    def __sub__(self, other): return self + (-other)
    def __radd__(self, other): return self + other
    def __rmul__(self, other): return self * other

    def relu(self):
        out = Value(max(0, self.data), (self,), 'ReLU')
        def _bwd():
            self.grad += (out.data > 0) * out.grad
        out._backward = _bwd
        return out

    def __repr__(self):
        return f"Value(data={self.data:.4f}, grad={self.grad:.4f})"

print("Value 类加载完毕")

## 概念 1：拓扑排序保证梯度累积顺序

计算图是一个 DAG（有向无环图）。节点 `v` 的 `_backward()` 需要用到 `v` 的所有"下游节点"（消费了 `v.data` 的节点）的梯度，这些梯度必须先准备好。

拓扑排序给出一个线性顺序，使得：若 `u → v`（`u` 依赖 `v`），则 `u` 排在 `v` 后面。
反向遍历此顺序，就能保证每个节点 `_backward()` 被调用时，其上游的累积梯度已经完整。

DFS 后序遍历（post-order）天然生成拓扑序：先递归访问所有子节点，访问完后再把当前节点加入列表。

```
topo = []
visited = set()
def build(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:   # 向输入方向走
            build(child)
        topo.append(v)          # 后序：最后加入自己
build(output)
# topo[-1] == output，topo[0] 是叶节点
```

In [ ]:
# 演示拓扑排序顺序
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c   # L = a*b + c

topo = []
visited = set()
def build_topo(v):
    if v not in visited:
        visited.add(v)
        for child in v._prev:
            build_topo(child)
        topo.append(v)

build_topo(L)
print("拓扑序（从叶到输出）：")
for i, v in enumerate(topo):
    print(f"  [{i}] op={v._op!r:5s}  data={v.data:.2f}  prev_count={len(v._prev)}")
print()
print("逆序（backward 的遍历方向）：", [v._op or 'leaf' for v in reversed(topo)])

## 概念 2：从输出节点出发，`self.grad = 1.0`

损失 `L` 对自身的梯度恒为 1：`dL/dL = 1`。
这是链式法则的起点——所有后续的 `_backward()` 都在把"上游梯度"（即 `out.grad`）乘以局部导数后写入输入节点的 `.grad`。

```
self.grad = 1.0          # dL/dL = 1
for node in reversed(topo):
    node._backward()     # 链式传播
```

如果忘记设置 `self.grad = 1.0`，所有乘以 `out.grad` 的项都变成 0，梯度全部为零。

In [ ]:
# 演示：不设 grad=1.0 时梯度全零
a2 = Value(2.0); b2 = Value(3.0)
out2 = a2 * b2
# 手动调用 _backward，但不设置 out2.grad
out2._backward()
print("未设 out.grad=1.0 时：a2.grad =", a2.grad, "（预期 3.0，但得到 0）")

# 正确做法
a3 = Value(2.0); b3 = Value(3.0)
out3 = a3 * b3
out3.grad = 1.0          # ← 关键
out3._backward()
print("设置 out.grad=1.0 后：a3.grad =", a3.grad, "（预期 3.0）✅")

## 概念 3：梯度累积（`+=`）而非覆盖（`=`）

当同一个节点被多个后续节点使用时（菱形图），梯度来自多条路径，必须**相加**，不能用后一条路径覆盖前一条。

例：`L = a*a`（`a` 同时作为左右操作数）
- 路径1：`dL/d(左a) = a.data = 2`
- 路径2：`dL/d(右a) = a.data = 2`
- 总梯度：`a.grad = 2 + 2 = 4`（等于解析式 `dL/da = 2a = 4`）

在 `_backward` 里统一写 `self.grad += ...` 而非 `self.grad = ...`，自动处理菱形图。

In [ ]:
# 菱形图演示：L = a * a
a4 = Value(2.0)
L4 = a4 * a4          # a4 同时是 _prev 里的两个"子节点"（同一对象）
L4.grad = 1.0
L4._backward()         # 执行一次 mul 的 _backward
print(f"a4.grad = {a4.grad:.1f}  （预期 4.0，因为 d(a^2)/da = 2a = 4）")
# 注意：mul 的 _backward 用 +=，所以两条路径都累积进来了

## 1. ✏️ 实现 `Value.backward(self)`

**推理路线**：
1. 用 DFS 后序遍历从 `self` 出发建立拓扑序列 `topo`，`visited` 集合防止重复访问
2. 将 `self.grad` 置为 `1.0`（输出节点对自身的梯度）
3. 逆序遍历 `topo`，对每个节点调用 `node._backward()`，让梯度沿图向后流动

**参考输入输出**：

```
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c          # L.data = 5.0
L.backward()
# dL/da = b.data = 3.0
# dL/db = a.data = 2.0
# dL/dc = 1.0（加法对 c 的导数）
assert a.grad == 3.0 and b.grad == 2.0 and c.grad == 1.0
```

<details><summary>点击查看参考实现</summary>

```python
def backward(self):
    topo = []
    visited = set()
    def build(v):
        if v not in visited:
            visited.add(v)
            for child in v._prev:
                build(child)
            topo.append(v)
    build(self)
    self.grad = 1.0
    for node in reversed(topo):
        node._backward()
```

</details>

In [ ]:
def backward(self):
    # ✏️ TODO 步骤1：DFS 后序遍历，建立拓扑序列 topo
    topo = []
    visited = set()
    def build(v):
        pass  # ✏️ TODO：递归访问 v._prev，后序加入 topo

    build(self)

    # ✏️ TODO 步骤2：设置输出节点梯度为 1.0
    # self.grad = ???

    # ✏️ TODO 步骤3：逆序调用每个节点的 _backward()
    pass

# 将方法绑定到 Value 类
import types
Value.backward = types.MethodType.__func__.__get__ if False else lambda cls, fn: setattr(cls, 'backward', fn) or cls
Value.backward = backward  # 直接替换

In [ ]:
# 检查：L = a*b + c
a = Value(2.0); b = Value(3.0); c = Value(-1.0)
L = a * b + c
L.backward()

assert abs(a.grad - 3.0) < 1e-9, f"a.grad 应为 3.0，得到 {a.grad}"
assert abs(b.grad - 2.0) < 1e-9, f"b.grad 应为 2.0，得到 {b.grad}"
assert abs(c.grad - 1.0) < 1e-9, f"c.grad 应为 1.0，得到 {c.grad}"
print(f"a.grad={a.grad:.1f}  b.grad={b.grad:.1f}  c.grad={c.grad:.1f}")
print("✅ backward() 基本检查通过")

# 菱形图检查：L = a*a，dL/da = 2*a = 4
a2 = Value(2.0)
L2 = a2 * a2
L2.backward()
assert abs(a2.grad - 4.0) < 1e-9, f"a2.grad 应为 4.0，得到 {a2.grad}"
print(f"菱形图 a2.grad={a2.grad:.1f}（预期 4.0）")
print("✅ 菱形图梯度累积检查通过")

## 参数实验：数值微分验证梯度

用数值微分 `(f(x+h) - f(x-h)) / (2h)` 独立计算梯度，与 `backward()` 结果对比，差距应 `< 1e-4`。

实验参数：
- `h = 1e-5`（步长太大误差大，太小浮点抖动）
- 测试函数：`L = a*b + c.relu()`（含 relu 的混合表达式）
- 预期现象：`a.grad` 等于 `b.data`，数值微分与解析梯度差距 `< 1e-4`

In [ ]:
def numeric_grad(f, x, h=1e-5):
    """对标量输入 x 计算 f 的数值梯度（中心差分）。"""""
    return (f(x + h) - f(x - h)) / (2 * h)

# 固定其他参数，逐一扰动输入
a_val, b_val, c_val = 2.0, 3.0, -0.5

def L_a(av): return (Value(av) * Value(b_val) + Value(c_val).relu()).data
def L_b(bv): return (Value(a_val) * Value(bv) + Value(c_val).relu()).data
def L_c(cv): return (Value(a_val) * Value(b_val) + Value(cv).relu()).data

num_a = numeric_grad(L_a, a_val)
num_b = numeric_grad(L_b, b_val)
num_c = numeric_grad(L_c, c_val)

# 解析梯度（backward）
a = Value(a_val); b = Value(b_val); c = Value(c_val)
L = a * b + c.relu()
L.backward()

print(f"{'变量':<6} {'backward':>12} {'数值微分':>12} {'误差':>12}")
for name, ana, num in [('a', a.grad, num_a), ('b', b.grad, num_b), ('c', c.grad, num_c)]:
    err = abs(ana - num)
    flag = "✅" if err < 1e-4 else "❌"
    print(f"{name:<6} {ana:>12.6f} {num:>12.6f} {err:>12.2e} {flag}")

## 小结

`Value.backward()` 通过 DFS 后序遍历建立拓扑序，再逆序调用每个节点的 `_backward()`，输出每个叶节点的 `.grad`（即损失对该参数的偏导数）。
梯度用 `+=` 累积，保证菱形图（参数复用）下多路径贡献正确相加。
此机制直接对应 `aurora/train/` 模块的参数更新步骤：`w.data -= lr * w.grad`。
下一节（M2-A4）将在此基础上组装 `MLP`，把多层 `Value` 节点串联成真实网络，并跑一次完整的前向 + 反向 + 更新循环。